# Fuzzy Logic Concrete Mix Workability Advisor
**L.D. College of Engineering, Ahmedabad — Gujarat Technological University**

**Mamdani Fuzzy Inference System (scikit-fuzzy)**

This notebook follows the project presentation slide-by-slide and implements the **exact** Fuzzy Inference
System shown in the PDF: 5 linguistic terms per variable (mixed trapezoidal / triangular / Gaussian MFs),
the 8 authored IF–THEN rules (R1–R8), Mamdani min/max inference and weighted-average defuzzification.

Run each cell in order (Shift+Enter).

## Slide 2 — Why Does Concrete Mix Design Need Fuzzy Logic?

**Scenario:** A ready-mix concrete plant wants to predict workability automatically before a pour.
A simple rule like *"if water–cement ratio < 0.40, reject the mix"* ignores that a lean mix with
fine aggregate on a cool day can place perfectly well, while the same ratio with coarse aggregate
on a hot day stiffens before it reaches the site. Site engineers reason in vague terms —
*"fairly dry," "moderately coarse," "quite hot"* — not sharp thresholds. Fuzzy logic lets a
quality-control controller reason the same way.

**Design Task:** Build a fuzzy inference system that reads Water–Cement Ratio, Aggregate Size,
and Ambient Temperature, and outputs a Workability Index — without a single hard-coded threshold.

**Learning Objectives:**
- Differentiate crisp sets from fuzzy sets
- Design membership functions for real variables
- Fuzzify crisp mix-design readings
- Build a fuzzy IF–THEN rule base
- Apply Mamdani inference (min / max)
- Aggregate rule outputs and defuzzify (weighted average)
- Evaluate fuzzy logic vs. threshold-based acceptance

## Step 1 — Setup: Install & Import Libraries

In [1]:
!pip install -q scikit-fuzzy plotly anywidget ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import pandas as pd
import plotly.graph_objects as go

# Colab: needed so FigureWidget / ipywidgets sliders actually respond
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

print("Libraries loaded successfully.")

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Libraries loaded successfully.


## Slide 3 — Crisp Sets vs. Fuzzy Sets

A **crisp set** gives a value either full membership (1) or none (0) — e.g. "water–cement ratio
< 0.40 = Low, otherwise not Low." A **fuzzy set** allows partial, graded membership between 0 and 1,
so a ratio of 0.42 can be "somewhat Low" and "somewhat Medium" at the same time. This matches how
site engineers actually describe mixes, and it's the foundation the rest of this notebook builds on.

In [2]:
# Quick illustration: crisp vs fuzzy membership for "Low" water-cement ratio
x = np.arange(0.30, 0.70, 0.001)

# Crisp version: hard threshold at 0.40
crisp_low = (x < 0.40).astype(float)

# Fuzzy version: triangular membership (defined fully in Step 3 below)
fuzzy_low = fuzz.trimf(x, [0.30, 0.30, 0.45])

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=crisp_low, mode='lines', name='Crisp "Low" (hard threshold)',
                          line=dict(color='#C0392B', width=3, dash='dash')))
fig.add_trace(go.Scatter(x=x, y=fuzzy_low, mode='lines', name='Fuzzy "Low" (graded)',
                          line=dict(color='#2E86AB', width=3)))
fig.update_layout(title='Crisp Sets vs. Fuzzy Sets (zoom / pan / hover enabled)',
                   xaxis_title='Water-Cement Ratio', yaxis_title='Membership degree',
                   template='plotly_white', hovermode='x unified')
fig.show()

## Slide 4 — Architecture of a Fuzzy Inference System

A Mamdani FIS has four stages, all implemented step-by-step in this notebook:

1. **Fuzzification** — convert crisp inputs into fuzzy membership degrees (Steps 3–6, 9)
2. **Rule Evaluation** — apply the IF–THEN rule base using min (AND) (Steps 7, 10)
3. **Aggregation & Implication** — clip and combine fired rule outputs (Step 11 / `skfuzzy` internally)
4. **Defuzzification** — convert the aggregated fuzzy output back into one crisp number, the
   Workability Index (Steps 11–12)

## Slide 5 — Inputs, Outputs & Range

**3 Inputs → 1 Output.** The system reads Water–Cement Ratio, Aggregate Size, and Ambient
Temperature, and outputs a single Workability Index.

## Step 2 — Define Inputs/Outputs & Universe of Discourse

In [3]:
wc_ratio = ctrl.Antecedent(np.arange(0.3, 0.7001, 0.001), 'wc_ratio')
agg_size = ctrl.Antecedent(np.arange(5, 40.01, 0.1), 'agg_size')
temp     = ctrl.Antecedent(np.arange(10, 45.01, 0.1), 'temp')
workability = ctrl.Consequent(np.arange(0, 100.01, 0.1), 'workability')

print("Water-Cement Ratio range:", wc_ratio.universe.min(), "-", wc_ratio.universe.max())
print("Aggregate Size range:", agg_size.universe.min(), "-", agg_size.universe.max())
print("Temperature range:", temp.universe.min(), "-", temp.universe.max())
print("Workability Index range:", workability.universe.min(), "-", workability.universe.max())

Water-Cement Ratio range: 0.3 - 0.7000000000000004
Aggregate Size range: 5.0 - 39.99999999999987
Temperature range: 10.0 - 44.99999999999987
Workability Index range: 0.0 - 100.0


## Slide 6 — Linguistic Variables

| Variable | Linguistic Terms |
|---|---|
| Water–Cement Ratio (0.30–0.70) | Very Low, Low, Medium, High, Very High |
| Aggregate Size (5–40 mm) | Very Fine, Fine, Medium, Coarse, Very Coarse |
| Ambient Temperature (10–45°C) | Very Cold, Cold, Moderate, Hot, Very Hot |
| Workability Index (0–100, output) | Very Poor, Poor, Acceptable, Good, Excellent |

**Total = 4 × 5 = 20 membership functions.**

## Slide 7 — Membership Functions: Water–Cement Ratio

Range: 0.30–0.70

| Term | MF | Parameters |
|---|---|---|
| Very Low | Trapezoidal | (0.30, 0.30, 0.34, 0.38) |
| Low | Triangular | (0.34, 0.40, 0.46) |
| Medium | Gaussian | centre = 0.50, σ = 0.06 |
| High | Triangular | (0.54, 0.62, 0.68) |
| Very High | Trapezoidal | (0.64, 0.68, 0.70, 0.70) |

## Step 3 — Membership Functions: Water–Cement Ratio

In [4]:
wc_terms = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
wc_ratio['Very Low']  = fuzz.trapmf(wc_ratio.universe, [0.30, 0.30, 0.34, 0.38])
wc_ratio['Low']       = fuzz.trimf(wc_ratio.universe,  [0.34, 0.40, 0.46])
wc_ratio['Medium']    = fuzz.gaussmf(wc_ratio.universe, 0.50, 0.06)
wc_ratio['High']      = fuzz.trimf(wc_ratio.universe,  [0.54, 0.62, 0.68])
wc_ratio['Very High'] = fuzz.trapmf(wc_ratio.universe, [0.64, 0.68, 0.70, 0.70])

def plot_mf(var, terms, title, xlabel):
    colors = ['#8E1B1B', '#C0392B', '#E67E22', '#27AE60', '#2E86AB']
    fig = go.Figure()
    for t, col in zip(terms, colors):
        fig.add_trace(go.Scatter(x=var.universe, y=var[t].mf, mode='lines', name=t, line=dict(color=col, width=3)))
    fig.update_layout(title=title + ' (zoom / pan / hover enabled)', xaxis_title=xlabel,
                      yaxis_title='Membership degree μ(x)', template='plotly_white', hovermode='x unified')
    fig.show()

plot_mf(wc_ratio, wc_terms, 'Membership Functions — Water-Cement Ratio', 'Water-Cement Ratio')

## Slide 8 — Membership Functions: Aggregate Size

Range: 5–40 mm

| Term | MF | Parameters |
|---|---|---|
| Very Fine | Trapezoidal | (5, 5, 7, 10) |
| Fine | Triangular | (7, 12, 18) |
| Medium | Gaussian | centre = 20, σ = 5 |
| Coarse | Triangular | (22, 30, 36) |
| Very Coarse | Trapezoidal | (34, 37, 40, 40) |

**Interpretation:** Smaller aggregate → Fine/Very Fine · Middle range → Medium · Larger aggregate → Coarse/Very Coarse.

## Step 4 — Membership Functions: Aggregate Size

In [5]:
agg_terms = ['Very Fine', 'Fine', 'Medium', 'Coarse', 'Very Coarse']
agg_size['Very Fine']   = fuzz.trapmf(agg_size.universe, [5, 5, 7, 10])
agg_size['Fine']        = fuzz.trimf(agg_size.universe,  [7, 12, 18])
agg_size['Medium']      = fuzz.gaussmf(agg_size.universe, 20, 5)
agg_size['Coarse']      = fuzz.trimf(agg_size.universe,  [22, 30, 36])
agg_size['Very Coarse'] = fuzz.trapmf(agg_size.universe, [34, 37, 40, 40])

plot_mf(agg_size, agg_terms, 'Membership Functions — Aggregate Size', 'Aggregate Size (mm)')

## Slide 9 — Membership Functions: Ambient Temperature

Range: 10–45°C. As temperature increases, concrete may lose workability faster.

| Term | MF | Parameters |
|---|---|---|
| Very Cold | Trapezoidal | (10, 10, 12, 16) |
| Cold | Triangular | (13, 18, 23) |
| Moderate | Gaussian | centre = 27, σ = 5 |
| Hot | Triangular | (30, 36, 42) |
| Very Hot | Trapezoidal | (40, 43, 45, 45) |

## Step 5 — Membership Functions: Ambient Temperature

In [6]:
tmp_terms = ['Very Cold', 'Cold', 'Moderate', 'Hot', 'Very Hot']
temp['Very Cold'] = fuzz.trapmf(temp.universe, [10, 10, 12, 16])
temp['Cold']      = fuzz.trimf(temp.universe,  [13, 18, 23])
temp['Moderate']  = fuzz.gaussmf(temp.universe, 27, 5)
temp['Hot']       = fuzz.trimf(temp.universe,  [30, 36, 42])
temp['Very Hot']  = fuzz.trapmf(temp.universe, [40, 43, 45, 45])

plot_mf(temp, tmp_terms, 'Membership Functions — Ambient Temperature', 'Ambient Temperature (°C)')

## Slide 10–11 — Membership Functions: Workability Index (Output)

Range: 0–100

| Term | MF | Parameters | Peak used for weighted average |
|---|---|---|---|
| Very Poor | Trapezoidal | (0, 0, 10, 20) | 0 |
| Poor | Triangular | (10, 25, 40) | 25 |
| Acceptable | Gaussian | centre = 45, σ = 10 | 45 |
| Good | Triangular | (55, 70, 85) | **70** |
| Excellent | Trapezoidal | (80, 90, 100, 100) | 100 |

## Step 6 — Membership Functions: Output (Workability Index)

In [7]:
out_terms = ['Very Poor', 'Poor', 'Acceptable', 'Good', 'Excellent']
workability['Very Poor']  = fuzz.trapmf(workability.universe, [0, 0, 10, 20])
workability['Poor']       = fuzz.trimf(workability.universe,  [10, 25, 40])
workability['Acceptable'] = fuzz.gaussmf(workability.universe, 45, 10)
workability['Good']       = fuzz.trimf(workability.universe,  [55, 70, 85])
workability['Excellent']  = fuzz.trapmf(workability.universe, [80, 90, 100, 100])

plot_mf(workability, out_terms, 'Membership Functions — Workability Index (Output)', 'Workability Index')

## Slide 12 — Fuzzy Rule Base

**Rule format:** IF W/C is *Medium* AND Aggregate is *Medium* AND Temperature is *Moderate*
THEN Workability is *Good*.

| Rule | Water–Cement | Aggregate | Temperature | Workability |
|---|---|---|---|---|
| R1 | Very Low | Very Coarse | Very Hot | Very Poor |
| R2 | Low | Fine | Moderate | Acceptable |
| R3 | Medium | Medium | Moderate | Good |
| R4 | High | Fine | Cold | Excellent |
| R5 | High | Coarse | Hot | Good |
| R6 | Medium | Medium | Hot | Acceptable |
| R7 | Medium | Medium | Cold | Good |
| R8 | Low | Fine | Cold | Acceptable |

Only these 8 rules from the presentation are used — **no extra rules are added**. AND is combined using the minimum operator (Mamdani, Zadeh AND).

## Step 7 — Build the Fuzzy Rule Base (Full 27 Rules)

In [8]:
# Peak value of each output term (used by the weighted-average defuzzification)
PEAK = {'Very Poor': 0, 'Poor': 25, 'Acceptable': 45, 'Good': 70, 'Excellent': 100}

# The 8 rules exactly as given in the presentation (slide 12). No extra rules are added.
authored = [
    ('Very Low', 'Very Coarse', 'Very Hot',  'Very Poor'),   # R1
    ('Low',      'Fine',        'Moderate',  'Acceptable'),  # R2
    ('Medium',   'Medium',      'Moderate',  'Good'),        # R3
    ('High',     'Fine',        'Cold',      'Excellent'),   # R4
    ('High',     'Coarse',      'Hot',       'Good'),        # R5
    ('Medium',   'Medium',      'Hot',       'Acceptable'),  # R6
    ('Medium',   'Medium',      'Cold',      'Good'),        # R7
    ('Low',      'Fine',        'Cold',      'Acceptable'),  # R8
]

rule_table = [(a, b, c, o, True) for a, b, c, o in authored]   # (wc, agg, temp, output, authored?)
rules = [ctrl.Rule(wc_ratio[a] & agg_size[b] & temp[c], workability[o]) for a, b, c, o, _ in rule_table]

for n, (a, b, c, o, _) in enumerate(rule_table, 1):
    print(f"R{n} IF W/C={a:<9} AND Agg={b:<11} AND Temp={c:<9} THEN Workability={o}")
print(f"\nTotal rules: {len(rules)}")

R1 IF W/C=Very Low  AND Agg=Very Coarse AND Temp=Very Hot  THEN Workability=Very Poor
R2 IF W/C=Low       AND Agg=Fine        AND Temp=Moderate  THEN Workability=Acceptable
R3 IF W/C=Medium    AND Agg=Medium      AND Temp=Moderate  THEN Workability=Good
R4 IF W/C=High      AND Agg=Fine        AND Temp=Cold      THEN Workability=Excellent
R5 IF W/C=High      AND Agg=Coarse      AND Temp=Hot       THEN Workability=Good
R6 IF W/C=Medium    AND Agg=Medium      AND Temp=Hot       THEN Workability=Acceptable
R7 IF W/C=Medium    AND Agg=Medium      AND Temp=Cold      THEN Workability=Good
R8 IF W/C=Low       AND Agg=Fine        AND Temp=Cold      THEN Workability=Acceptable

Total rules: 8


## Step 8 — Build the Control System

In [9]:
workability_ctrl = ctrl.ControlSystem(rules)
sim = ctrl.ControlSystemSimulation(workability_ctrl)

def categorize(v):
    """Class = output term with the highest membership at v (no hard-coded cut-offs)."""
    mus = {t: float(fuzz.interp_membership(workability.universe, workability[t].mf, v)) for t in out_terms}
    return max(mus, key=mus.get)

def evaluate_rules(x_wc, x_agg, x_tp):
    """Fuzzify the inputs and return per-rule firing strengths (min operator)."""
    mu_wc  = {t: float(fuzz.interp_membership(wc_ratio.universe,  wc_ratio[t].mf,  x_wc))  for t in wc_terms}
    mu_agg = {t: float(fuzz.interp_membership(agg_size.universe, agg_size[t].mf, x_agg)) for t in agg_terms}
    mu_tmp = {t: float(fuzz.interp_membership(temp.universe,     temp[t].mf,     x_tp))  for t in tmp_terms}
    rows = []
    for n, (a, b, c, o, _) in enumerate(rule_table, 1):
        w = min(mu_wc[a], mu_agg[b], mu_tmp[c])
        rows.append(dict(rule=f"R{n}", a=mu_wc[a], b=mu_agg[b], c=mu_tmp[c], w=w, out=o,
                         label=f"{a}/{b}/{c}"))
    return rows

def aggregated_curve(rows):
    curve = np.zeros_like(workability.universe)
    for r in rows:
        if r['w'] > 1e-9:
            curve = np.fmax(curve, np.fmin(r['w'], workability[r['out']].mf))
    return curve

print("Control system and simulation ready.")

Control system and simulation ready.


## Slide 13–14 — Fuzzification: Sample Data

Take one sample concrete mix:
- **W/C Ratio = 0.42**
- **Aggregate = 18 mm**
- **Temperature = 30°C**

Expected (from the slides): W/C → Low = 0.667, Medium = 0.411 · Aggregate → Medium = 0.923 · Temperature → Moderate = 0.835

## Step 9 — Manual Fuzzification Walkthrough

In [10]:
x_wc, x_agg, x_temp = 0.42, 18, 30

def trimf_manual(x, a, b, c):
    left = (x - a) / (b - a) if b != a else 1
    right = (c - x) / (c - b) if c != b else 1
    return max(min(left, right), 0)

def trapmf_manual(x, a, b, c, d):
    left = (x - a) / (b - a) if b != a else 1
    right = (d - x) / (d - c) if d != c else 1
    return max(min(left, 1, right), 0)

def gaussmf_manual(x, c, s):
    return float(np.exp(-0.5 * ((x - c) / s) ** 2))

mu_wc = {
    'Very Low':  trapmf_manual(x_wc, 0.30, 0.30, 0.34, 0.38),
    'Low':       trimf_manual(x_wc, 0.34, 0.40, 0.46),
    'Medium':    gaussmf_manual(x_wc, 0.50, 0.06),
    'High':      trimf_manual(x_wc, 0.54, 0.62, 0.68),
    'Very High': trapmf_manual(x_wc, 0.64, 0.68, 0.70, 0.70),
}
mu_ag = {
    'Very Fine':   trapmf_manual(x_agg, 5, 5, 7, 10),
    'Fine':        trimf_manual(x_agg, 7, 12, 18),
    'Medium':      gaussmf_manual(x_agg, 20, 5),
    'Coarse':      trimf_manual(x_agg, 22, 30, 36),
    'Very Coarse': trapmf_manual(x_agg, 34, 37, 40, 40),
}
mu_tp = {
    'Very Cold': trapmf_manual(x_temp, 10, 10, 12, 16),
    'Cold':      trimf_manual(x_temp, 13, 18, 23),
    'Moderate':  gaussmf_manual(x_temp, 27, 5),
    'Hot':       trimf_manual(x_temp, 30, 36, 42),
    'Very Hot':  trapmf_manual(x_temp, 40, 43, 45, 45),
}
print("W/C = 0.42   ->", {k: round(v, 3) for k, v in mu_wc.items()})
print("Agg = 18 mm  ->", {k: round(v, 3) for k, v in mu_ag.items()})
print("Temp = 30 C  ->", {k: round(v, 3) for k, v in mu_tp.items()})

W/C = 0.42   -> {'Very Low': 0, 'Low': 0.667, 'Medium': 0.411, 'High': 0, 'Very High': 0}
Agg = 18 mm  -> {'Very Fine': 0, 'Fine': 0.0, 'Medium': 0.923, 'Coarse': 0, 'Very Coarse': 0}
Temp = 30 C  -> {'Very Cold': 0, 'Cold': 0, 'Moderate': 0.835, 'Hot': 0.0, 'Very Hot': 0}


## Slide 15 — Rule Firing: Mamdani Inference

**Inference Process:**
1. **Fuzzification** — convert input values into membership degrees.
2. **Rule Evaluation** — calculate firing strength using w = min(μ₁, μ₂, μ₃).
3. **Implication** — clip the output membership function at the firing strength:
   μ'ₒᵤₜ(y) = min(w, μₒᵤₜ(y))
4. **Aggregation** — combine all fired rules using: μ_agg(y) = max(μ'₁, μ'₂, ..., μ'ₙ)

**Fired rule example — R3:**
IF W/C = Medium (μ = 0.411) AND Aggregate = Medium (μ = 0.923) AND Temperature = Moderate (μ = 0.835)
→ AND = minimum → w₃ = min(0.411, 0.923, 0.835) = **0.411**
→ THEN Workability is Good, R3 fires with strength 0.411.

## Step 10 — Identify Fired Rules & Firing Strength
A rule fires only if **every** antecedent term has non-zero membership. AND = minimum.

In [11]:
rows_sample = evaluate_rules(x_wc, x_agg, x_temp)

fired_rules = []
for r in rows_sample:
    if r['w'] > 1e-9:   # ignore floating-point noise (exact zeros)
        fired_rules.append((r['rule'], r['w'], r['out'], PEAK[r['out']]))
        print(f"{r['rule']:<3} FIRES ({r['label']}) -> w = min({r['a']:.3f}, {r['b']:.3f}, {r['c']:.3f}) = {r['w']:.3f}  =>  {r['out']}")
print(f"\n{len(fired_rules)} of {len(rule_table)} rules fire for this mix.")

R3  FIRES (Medium/Medium/Moderate) -> w = min(0.411, 0.923, 0.835) = 0.411  =>  Good

1 of 8 rules fire for this mix.


## Slide 16 — Defuzzification

**Weighted Average Method** (for classroom calculation):

y* = Σ(wᵢ · peakᵢ) / Σ(wᵢ)

For this sample mix: y* = (0.411 × 70) / 0.411 = **70** → **Workability Index = 70, Classification: Good**.

Unlike a crisp threshold system, fuzzy logic provides a graded engineering decision by
considering multiple uncertain parameters simultaneously.

## Step 11 — Weighted-Average Defuzzification (Manual)
y* = Σ(wᵢ · peakᵢ) / Σ(wᵢ)

In [12]:
sum_w = sum(w for _, w, _, _ in fired_rules)
sum_w_peak = sum(w * peak for _, w, _, peak in fired_rules)
y_star_manual = sum_w_peak / sum_w

print(f"Sum(w)        = {sum_w:.3f}")
print(f"Sum(w*peak)   = {sum_w_peak:.3f}")
print(f"y* (manual)   = {sum_w_peak:.3f} / {sum_w:.3f} = {y_star_manual:.2f}  (out of 100)")
print(f"Classification: {categorize(y_star_manual)}")

Sum(w)        = 0.411
Sum(w*peak)   = 28.778
y* (manual)   = 28.778 / 0.411 = 70.00  (out of 100)
Classification: Good


## Step 12 — Run scikit-fuzzy Simulation & Verify

In [13]:
sim.input['wc_ratio'] = x_wc
sim.input['agg_size'] = x_agg
sim.input['temp'] = x_temp
sim.compute()

print(f"scikit-fuzzy output (centroid)  = {sim.output['workability']:.2f}")
print(f"Manual weighted-average output  = {y_star_manual:.2f}")

# Interactive plot: output membership functions + aggregated (clipped) result + crisp output line
y_uni = workability.universe
agg = np.zeros_like(y_uni)
for name, w, out_term, peak in fired_rules:
    clipped = np.fmin(w, workability[out_term].mf)
    agg = np.fmax(agg, clipped)

fig = go.Figure()
for term, color in [('Very Poor','#8E1B1B'), ('Poor','#C0392B'), ('Acceptable','#E67E22'), ('Good','#27AE60'), ('Excellent','#2E86AB')]:
    fig.add_trace(go.Scatter(x=y_uni, y=workability[term].mf, mode='lines', name=term,
                              line=dict(color=color, width=1.5, dash='dot'), opacity=0.5))
fig.add_trace(go.Scatter(x=y_uni, y=agg, mode='lines', name='Aggregated output', fill='tozeroy',
                          line=dict(color='#8E44AD', width=3)))
fig.add_vline(x=sim.output['workability'], line=dict(color='black', width=2, dash='dash'),
              annotation_text=f"Crisp output = {sim.output['workability']:.2f}")
fig.update_layout(title='Mamdani Aggregation & Defuzzification (zoom / pan / hover enabled)',
                   xaxis_title='Workability Index', yaxis_title='Membership degree',
                   template='plotly_white', hovermode='x unified')
fig.show()

scikit-fuzzy output (centroid)  = 70.00
Manual weighted-average output  = 70.00


## Step 13 — Test 17 Different Input Combinations
Going beyond the single sample from the slides, to sanity-check the controller across a spread of inputs.

In [14]:
test_inputs = [
    (0.32, 8, 15),
    (0.42, 18, 30),
    (0.55, 12, 20),
    (0.65, 35, 40),
    (0.38, 10, 12),
    (0.50, 20, 25),
    (0.45, 30, 38),
    (0.60, 8, 18),
    (0.35, 25, 33),
    (0.68, 38, 42),
    (0.31, 6, 11),
    (0.48, 22, 27),
    (0.58, 16, 34),
    (0.40, 14, 21),
    (0.63, 28, 44),
    (0.62, 32, 38),
    (0.36, 19, 29),
]

rows = []
for wc, agg, tp in test_inputs:
    s = ctrl.ControlSystemSimulation(workability_ctrl)
    s.input['wc_ratio'] = wc
    s.input['agg_size'] = agg
    s.input['temp'] = tp
    s.compute()
    out = s.output['workability']
    max_w = max(r['w'] for r in evaluate_rules(wc, agg, tp))
    rows.append([wc, agg, tp, round(out, 2), categorize(out), round(max_w, 3)])

df = pd.DataFrame(rows, columns=["WC Ratio", "Agg Size (mm)", "Temp (°C)", "Workability Index", "Category", "Max rule strength"])
df

    WC Ratio  Agg Size (mm)  Temp (°C)  Workability Index    Category  \
0       0.32              8         15              70.00        Good   
1       0.42             18         30              70.00        Good   
2       0.55             12         20              74.78        Good   
3       0.65             35         40              66.54        Good   
4       0.38             10         12              48.42  Acceptable   
5       0.50             20         25              70.00        Good   
6       0.45             30         38              49.96  Acceptable   
7       0.60              8         18              84.41   Excellent   
8       0.35             25         33              50.50  Acceptable   
9       0.68             38         42              70.00        Good   
10      0.31              6         11              70.00        Good   
11      0.48             22         27              70.00        Good   
12      0.58             16         34             

All **17 test combinations** above are pre-computed and shown in the table for reference.
The **Max rule strength** column shows how strongly the best rule fired — the 8-rule base only covers part of the input space, so a small value means the result is weakly supported.

Below, you can **pick any one of these 17 cases from a dropdown, or type in your own custom values** —
nothing else is shown until you make a selection. Once you choose/submit, you'll see:
- A **breakdown table** (fuzzification, rule firing, Mamdani weighted values) for that single case only
- A **live interactive graph** (aggregated output + crisp Workability Index) for that single case only

In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output

case_options = [("-- Select a case --", None)] + [
    (f"Case {i+1}: WC={wc}, Agg={agg}mm, Temp={tp}°C", i)
    for i, (wc, agg, tp) in enumerate(test_inputs)
]
case_dropdown = widgets.Dropdown(options=case_options, description='Pick case:', style={'description_width': 'initial'})
case_output = widgets.Output()

def show_case_detail(wc, agg, tp, label):
    """Computes and displays ONE breakdown table + ONE live graph for the given inputs."""
    rows = evaluate_rules(wc, agg, tp)
    fired = [r for r in rows if r['w'] > 1e-3]   # display filter only (hides negligible Gaussian tails)
    detail_df = pd.DataFrame(
        [[r['rule'], r['label'], round(r['a'],3), round(r['b'],3), round(r['c'],3), round(r['w'],3), r['out']] for r in fired],
        columns=["Rule", "WC / Agg / Temp", "μ(WC)", "μ(Agg)", "μ(Temp)", "Firing w=min(...)", "Output Term"])
    agg_curve = aggregated_curve(rows)

    s = ctrl.ControlSystemSimulation(workability_ctrl)
    s.input['wc_ratio'] = wc
    s.input['agg_size'] = agg
    s.input['temp'] = tp
    s.compute()
    crisp_val = s.output['workability']
    cat = categorize(crisp_val)

    print(f"Selected: {label}")
    print(f"Inputs -> WC Ratio: {wc}, Aggregate: {agg} mm, Temp: {tp}°C")
    print(f"Fired rules: {len(fired)} of {len(rows)}")
    display(detail_df)
    print(f"\nMamdani crisp output (centroid) = {crisp_val:.2f} / 100  ->  {cat}")

    fig = go.Figure()
    for term, color in [('Very Poor','#8E1B1B'), ('Poor','#C0392B'), ('Acceptable','#E67E22'), ('Good','#27AE60'), ('Excellent','#2E86AB')]:
        fig.add_trace(go.Scatter(x=workability.universe, y=workability[term].mf, mode='lines', name=term,
                                  line=dict(color=color, width=1.5, dash='dot'), opacity=0.5))
    fig.add_trace(go.Scatter(x=workability.universe, y=agg_curve, mode='lines', name='Aggregated output',
                              fill='tozeroy', line=dict(color='#8E44AD', width=3)))
    fig.add_vline(x=crisp_val, line=dict(color='black', width=2, dash='dash'),
                  annotation_text=f"Crisp output = {crisp_val:.2f}")
    fig.update_layout(title=f'Mamdani Output — {label}',
                       xaxis_title='Workability Index', yaxis_title='Membership degree',
                       template='plotly_white', hovermode='x unified', height=420)
    fig.show()

def on_case_change(change):
    with case_output:
        clear_output(wait=True)
        idx = change['new']
        if idx is None:
            print("Select a case above to see its breakdown table and graph.")
            return
        wc, agg, tp = test_inputs[idx]
        show_case_detail(wc, agg, tp, f"Case {idx+1}")

case_dropdown.observe(on_case_change, names='value')

display(case_dropdown, case_output)
with case_output:
    print("Select a case above to see its breakdown table and graph.")

### Or Enter Your Own Custom Values

Type your own Water–Cement Ratio, Aggregate Size, and Temperature below, then click **Compute**.
Nothing is shown until you click the button — only your one result (table + graph) appears.

In [16]:
wc_box   = widgets.BoundedFloatText(value=0.42, min=0.30, max=0.70, step=0.01, description='WC Ratio:', style={'description_width': 'initial'})
agg_box  = widgets.BoundedFloatText(value=18, min=5, max=40, step=1, description='Agg Size (mm):', style={'description_width': 'initial'})
temp_box = widgets.BoundedFloatText(value=30, min=10, max=45, step=1, description='Temp (°C):', style={'description_width': 'initial'})
compute_btn = widgets.Button(description='Compute', button_style='primary', icon='check')
custom_output = widgets.Output()

def on_compute_click(b):
    with custom_output:
        clear_output(wait=True)
        show_case_detail(wc_box.value, agg_box.value, temp_box.value, "Custom Input")

compute_btn.on_click(on_compute_click)

display(widgets.HBox([wc_box, agg_box, temp_box, compute_btn]), custom_output)

## Step 14 — Analysis & Discussion

**Controller behaviour:** For the sample mix (0.42, 18 mm, 30 °C) only R3 fires (w = 0.411) and the
Workability Index is **70 → Good**, matching the hand calculation in the slides.

**Strengths:**
- Handles vague, real-world descriptions ("fairly dry", "quite hot") without hard cutoffs
- Produces graded, continuous output instead of binary accept/reject
- Rules are human-readable and can be inspected/tuned by a site engineer
- Balances multiple competing signals (lean mix vs. heat vs. coarse aggregate) simultaneously

**Weaknesses / limitations:**
- Only 8 rules cover a 5×5×5 = 125-combination input space, so many inputs are only weakly supported (see *Max rule strength*)
- Rule base and MF breakpoints were hand-tuned, not learned from real plant data
- Defuzzification choice (weighted average vs. centroid) can shift results when rules with different outputs fire together

**Scope for improvement:**
- Extend the rule base with a domain expert to cover more combinations
- Add a 4th input (Curing Humidity)
- Compare defuzzification methods (bisector, mean-of-maximum) against weighted-average
- Use a Genetic Algorithm to auto-tune membership function breakpoints instead of hand-tuning

## Step 15 — Interactive UI (Live Plotly + ipywidgets)
Drag the sliders below — the number **and the graph itself redraw in real time**
(a `go.FigureWidget` updates in place, unlike `plt.show()` which produces a static image
each run).

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

wc_slider   = widgets.FloatSlider(value=0.42, min=0.3, max=0.7, step=0.01, description='WC Ratio:')
agg_slider  = widgets.FloatSlider(value=18, min=5, max=40, step=1, description='Agg Size (mm):')
temp_slider = widgets.FloatSlider(value=30, min=10, max=45, step=1, description='Temp (°C):')

# FigureWidget = a live, in-place-updating Plotly figure (not a static image)
fig_widget = go.FigureWidget()
for term, color in [('Very Poor','#8E1B1B'), ('Poor','#C0392B'), ('Acceptable','#E67E22'), ('Good','#27AE60'), ('Excellent','#2E86AB')]:
    fig_widget.add_trace(go.Scatter(x=workability.universe, y=workability[term].mf, mode='lines',
                                     name=term, line=dict(color=color, width=1.5, dash='dot'), opacity=0.5))
fig_widget.add_trace(go.Scatter(x=workability.universe, y=np.zeros_like(workability.universe),
                                 mode='lines', name='Aggregated output', fill='tozeroy',
                                 line=dict(color='#8E44AD', width=3)))
fig_widget.update_layout(title='Live Mamdani Output (drag sliders below)',
                          xaxis_title='Workability Index', yaxis_title='Membership degree',
                          template='plotly_white', height=420)

result_label = widgets.HTML()

def update(change=None):
    try:
        s = ctrl.ControlSystemSimulation(workability_ctrl)
        s.input['wc_ratio'] = wc_slider.value
        s.input['agg_size'] = agg_slider.value
        s.input['temp'] = temp_slider.value
        s.compute()
        val = s.output['workability']
        cat = categorize(val)
        rows = evaluate_rules(wc_slider.value, agg_slider.value, temp_slider.value)
        with fig_widget.batch_update():
            fig_widget.data[-1].y = aggregated_curve(rows)
        result_label.value = f"<b>Workability Index: {val:.2f} / 100 &nbsp;&rarr;&nbsp; {cat}</b>"
    except Exception as e:
        result_label.value = f"<b style='color:#C0392B'>Error: {e}</b>"

wc_slider.observe(update, names='value')
agg_slider.observe(update, names='value')
temp_slider.observe(update, names='value')

display(wc_slider, agg_slider, temp_slider, result_label, fig_widget)
update()